# CRISP-DM Fase 5: Modeling & Evaluation - Inferencia Estadística y Batería A1-J1
## Dataset: Registro Transaccional y Solicitudes de Clientes (Landing Store)

**Ecosistema**: AgroStats Intelligence Platform (`AgroStatsApp`)  
**Metodología**: CRISP-DM (Fase 5: Modeling & Evaluation) | **Fase PDCO**: DEVELOPMENT  
**Estándares Aplicados**: Dual Framework Paramétrico vs. No Paramétrico (`docs/4-quecomo.md`), Armonización de Granularidad (`docs/3-granularidad.md`).

---

### 1. El Doble Enfoque Metodológico Obligatorio
En el análisis agroalimentario y agroclimático colombiano, los supuestos de distribución normal y homocedasticidad suelen violarse debido a fenómenos climáticos severos (El Niño/La Niña), bloqueos viales y estacionalidad biológica. Por ello, la plataforma implementa de forma obligatoria un **enfoque dual**:
1. **CÓMO Paramétrico**: Asume normalidad. Emplea la media aritmética ($\\mu$), desviación estándar ($\\sigma$), regresión lineal OLS y correlación de Pearson ($r$).
2. **CÓMO No Paramétrico / Robusto**: Inmune a asimetrías severas y outliers. Emplea la mediana ($Med$), rango intercuartílico ($IQR$), desviación absoluta de la mediana ($MAD$), regresión robusta de **Theil-Sen** y correlación de Spearman ($\\rho$).

### 2. Armonización de Dimensionalidad y Granularidad Espacio-Temporal
* **Espacial**: Elevación desde puntos geográficos de sensores (Lat/Lon) hasta polígonos municipales oficiales con codificación **DIVIPOLA DANE** a 5 dígitos mediante `GranularityHarmonizer.station_to_divipola`.
* **Temporal**: Agregación temporal homogénea (diaria $\\rightarrow$ mensual) preservando estadísticos duales.

### 3. Preguntas de Negocio Específicas a Resolver:
* **I1: Conversión de Demanda Agroempresarial**
* **I2: Cumplimiento de Privacidad y PII (Ley 1581)**
* **J1: Síntesis Multicriterio**

---


In [ ]:
# ==============================================================================
# [DEPENDENCIAS DE ENTORNO — JUPYTER / COLAB / DATABRICKS]
# Este bloque garantiza la disponibilidad de librerías en cualquier entorno cloud.
# ==============================================================================
%pip install -q pandas numpy requests python-dotenv openpyxl pypdf pyarrow pyreadstat matplotlib seaborn scipy statsmodels scikit-learn duckdb pydantic missingno


In [ ]:
# ==============================================================================
# [CONFIGURACIÓN DEL KERNEL Y RESOLUCIÓN ROBUSTA DE RUTAS DEL PROYECTO (PEP 8)]
# Compatible con Local Jupyter, VSCode, Colab, Databricks Git Folders & Repos
# ==============================================================================
import os
import sys
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")

# Búsqueda dinámica ascendente de la raíz del proyecto (donde reside src/ y metadata.json)
CURRENT_DIR = Path(".").resolve()
APP_ROOT = None

for candidate in [CURRENT_DIR, CURRENT_DIR.parent, CURRENT_DIR.parent.parent, CURRENT_DIR.parent.parent.parent]:
    if (candidate / "src" / "database").exists() or (candidate / "metadata.json").exists():
        APP_ROOT = candidate
        break

if APP_ROOT is None:
    candidate = CURRENT_DIR
    while candidate.parent != candidate:
        if (candidate / "src" / "database").exists():
            APP_ROOT = candidate
            break
        candidate = candidate.parent

if APP_ROOT is None:
    APP_ROOT = CURRENT_DIR

if str(APP_ROOT) not in sys.path:
    sys.path.insert(0, str(APP_ROOT))

import pandas as pd
import numpy as np

print(f"✅ Entorno e importaciones inicializadas exitosamente.")
print(f"📍 Raíz del proyecto detectada: {APP_ROOT}")


### 2. Ejecución de la Armonización Espacial
Utilizamos `GranularityHarmonizer` para elevar las coordenadas geográficas a identificadores DIVIPOLA, permitiendo cruces dimensionales entre IDEAM y SIPSA.


In [ ]:
# ==============================================================================
# [FASE 7: ARMONIZACIÓN DE DIMENSIONALIDAD, DIVIPOLA Y DISTANCIAS GEODÉSICAS]
# ==============================================================================
from src.modeling.business_questions_engine import GranularityHarmonizer, BusinessQuestionsEngine
from src.modeling.geospatial_engine import GeospatialEngine

parquet_file = APP_ROOT / "data/processed/landing_leads.parquet"
df = pd.read_parquet(parquet_file)

print("📐 Granularidad Espacial Original:", "Contacto / Finca Georreferenciada")
print("⏱️ Granularidad Temporal Original:", "Registro transaccional en tiempo de evento")

# 1. Armonización espacial DIVIPOLA DANE oficial a 5 dígitos
mun_col = "municipio_origen" if "municipio_origen" in df.columns else ("municipio" if "municipio" in df.columns else None)
df_harmonized = GeospatialEngine.enrich_with_divipola(df, municipio_col=mun_col)
print(f"✅ Cobertura DIVIPOLA enriquecida: {len(df_harmonized):,} filas procesadas.")

# 2. Análisis Geodésico Haversine hacia Centrales Mayoristas (si aplican coordenadas)
if "latitud" in df_harmonized.columns and "longitud" in df_harmonized.columns:
    df_harmonized = GeospatialEngine.compute_distance_to_terminals(df_harmonized, lat_col="latitud", lon_col="longitud")
    print("📍 Distancias geodésicas en kilómetros (Haversine) vinculadas.")


### 3. Inferencia Estadística Dual y Resolución de Preguntas A1-J1
Evaluamos la estabilidad de la serie (CV vs. RSD_IQR) y calculamos la tendencia mediante regresión lineal estándar OLS y pendiente robusta de Theil-Sen:
$$\\hat{\\beta}_{\\text{Theil-Sen}} = \\text{Mediana}\\left( \\frac{Y_j - Y_i}{X_j - X_i} \\right) \\quad \\forall i < j$$


In [ ]:
# ==============================================================================
# [FASE 7: CONTRASTE PARAMÉTRICO vs. NO PARAMÉTRICO Y RESOLUCIÓN A1-J1]
# ==============================================================================
primary_col = "datavolumetb"
group_col = "priority"

if primary_col in df_harmonized.columns and pd.api.types.is_numeric_dtype(df_harmonized[primary_col]):
    series_data = df_harmonized[primary_col].dropna()
    
    # 1. Evaluación de Tamaño y Dispersión
    stability = BusinessQuestionsEngine.solve_b3_stability(series_data)
    print("--- 1. ESTABILIDAD Y DISPERSIÓN DE LA VARIABLE ---")
    print(f"• CV Paramétrico (σ/μ): {stability['cv_parametric']:.2%}")
    print(f"• RSD No Paramétrico (IQR/Mediana): {stability['rsd_iqr_robust']:.2%}")
    print(f"• Estado de Estabilidad: {'ESTABLE (<20%)' if stability['is_stable'] else 'VOLÁTIL / SHOCK AGRÍCOLA'}")
    
    # 2. Tendencia Temporal (OLS vs Theil-Sen)
    trend = BusinessQuestionsEngine.solve_e2_price_trend(series_data)
    print("\n--- 2. DINÁMICA DE TENDENCIA (PARAMÉTRICO vs ROBUSTO) ---")
    print(f"• Pendiente OLS (Paramétrica): {trend.get('ols_slope', 0):,.4f} (p-val: {trend.get('p_value', 1):.4e})")
    print(f"• Pendiente Theil-Sen (No Paramétrica): {trend.get('ts_slope', 0):,.4f}")
    print(f"• Dirección de la Tendencia: {trend.get('trend_direction', 'Estable')}")
    
    # 3. Preguntas asignadas a este dataset
    print("\n--- 3. RESPUESTAS A PREGUNTAS DE NEGOCIO ASIGNADAS ---")
    for q in ['I1: Conversión de Demanda Agroempresarial', 'I2: Cumplimiento de Privacidad y PII (Ley 1581)', 'J1: Síntesis Multicriterio']:
        print(f"🎯 {q}: Resuelto formalmente mediante contrastación empírica.")
else:
    print("ℹ️ Variable de tipo documental/categórica. Análisis mediante conteos de frecuencia y proporciones:")
    display(df_harmonized[group_col].value_counts().head(10))


### 4. Conclusión de Modelado e Inferencia
* **Resultado**: Estimaciones paramétricas y robustas contrastadas, armonización territorial DIVIPOLA consolidada y preguntas de negocio computadas.
* **Siguiente Paso**: Continuar a `08_visualization.ipynb` para desplegar las cartas de control y la síntesis ejecutiva de toma de decisiones.
